# MODI-SYNTH · Baseline Reader — SPAN edition (final)

Run with **GPU T4 x2**, **Internet On**, then *Save Version → Save & Run All (Commit)*.

## What this does
It trains **SPAN** (Coquenet et al., ICDAR 2021) — the published method for reading a multi-line
paragraph when all you have is the paragraph's transcription. The whole patch goes in untouched;
there is no line cutting. **Two versions of the model train at the same time, one on each GPU:**

| | GPU | starts from | why it is here |
|---|---|---|---|
| `scratch` | 0 | random weights | the clean baseline the paper's claims are measured against |
| `iam_init` | 1 | the authors' IAM-trained encoder (CC-BY-4.0) | the paper reports pre-training speeds convergence; this tests whether English-handwriting features help on Modi |

Same data, same settings, same seed. Both GPUs are otherwise idle in a Kaggle session, so the second
model costs no extra quota.

## What was checked before this was handed to you
* the network matches the authors' trained network **exactly**: 84 tensors, identical names and
  shapes, 18.15 M parameters; their weights load with `strict=True`
* turning their 3-channel input into our 1-channel input is exact (max difference 1.4e-06)
* the whole pipeline — smoke test, resume, two processes side by side, loss, decoding, logging —
  was rehearsed end to end on CPU with a tiny network
* every image edit was rendered and inspected (`image-processing-audit` folder); one problem found
  there (augmentation leaving ~3% of pages unreadable) was corrected, and is the only deviation
  from SPAN's recipe

## What was NOT checked, so you know
* **that it learns Modi.** Nothing here has run on a GPU yet. The 20-patch smoke test is the first
  real evidence, and it gates the long run.
* **epoch time.** Estimated at roughly 4 minutes per epoch (about 165 epochs in 11 h) by scaling
  your own measured T4 timings; the first epoch prints the real figure and an updated projection.

## What the paper gives us to expect
SPAN trained from scratch on RIMES (1,400 paragraphs, the same data scale as our 1,430 patches)
shows a training loss that falls smoothly from the first epochs — no long stall at the
letter-frequency level that every earlier run of ours had — reaching about 1.2 by epoch 100.
Pre-training reaches the same loss roughly 2–3× sooner. It is an expectation, not a promise.

## How to read the run
Both models print one block per epoch, prefixed `[scratch]` / `[iam_init]`.
* **~20 epochs in (about 80 minutes):** compare `loss`. Whichever is lower is learning faster.
* the honest early signs are `loss` falling below ~3.0, `blank` coming down from ~100%, and
  `len` (predicted length) climbing toward the true length — before CER moves
* if **neither** has `loss` below ~3.0 by epoch 40, stop and send the log; do not wait out the session

## Running past 12 hours
`last.pt` is written every epoch. To continue: *Add Input* → this notebook's own previous output →
commit again. Cell 1 restores `runs/`, the smoke test is skipped, and each model resumes where it was.

## Recipe, with sources
| setting | value | source |
|---|---|---|
| optimiser | Adam, lr 1e-4, no scheduler, no weight decay | SPAN `main_pg_ctc.py`, paper 4.5 |
| batch size | 4 | paper 4.5 (the repo script says 8; the paper's figure is used) |
| dropout | 0.5 (MixDropout, one random place per block) | SPAN `models.py` |
| precision | fp32 | SPAN `use_amp: False` |
| augmentation | each at p = 0.2: resolution, perspective / elastic / projective, dilation-erosion, contrast, brightness, sign flip | SPAN config |
| input | whole patch, 320x880, standardised with the training set's mean and std | paper 4.2 |
| loss | CTC on the 2D grid, flattened row by row, per sample cropped to its valid region | paper 3.2, trainer |
| **deviation** | brightness / contrast factor floor 0.3 instead of 0.01 | our own image audit |
| not copied | augmentations re-implemented with OpenCV, so intensities are close, not identical | — |

> SPAN: Coquenet, Chatelain, Paquet, ICDAR 2021, arXiv:2102.08742. Code CeCILL-C
> (github.com/FactoDeepLearning/SPAN); this is a reimplementation. Weights CC-BY-4.0, zenodo 7244546.

In [ ]:
# Cell 1 - environment, GPUs, wall-clock budget, and pick up any earlier session's progress
import os, sys, time, json, re, math, random, csv, glob, shutil, zipfile, subprocess, hashlib, pathlib
T_START = time.time()
BUDGET_H = 11.0                       # Kaggle stops a commit at 12 h; stop training at 11 h so the last cells run
DEADLINE = T_START + BUDGET_H * 3600
def elapsed_h(): return (time.time() - T_START) / 3600

import torch
print('torch', torch.__version__)
smi = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print(smi or 'NO GPU FOUND')
N_GPU = len([l for l in smi.splitlines() if l.startswith('GPU')])
assert N_GPU >= 1, 'Turn on the GPU: Settings -> Accelerator -> GPU T4 x2'
if N_GPU == 1:
    print('NOTE: only one GPU is visible, so only the IAM-initialised model will train. '
          'Choose GPU T4 x2 to run both side by side.')
print(f'budget: {BUDGET_H} h of training, then it stops and saves')

# RESUME: if a previous commit's output is attached as an input dataset, copy its runs/ folders here.
# Each model then finds its own last.pt and carries straight on; a smoke test that already passed is skipped.
restored = 0
for cand in glob.glob('/kaggle/input/*/runs/*/last.pt'):
    dst = '/kaggle/working/runs/' + cand.split('/runs/', 1)[1]
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for f in glob.glob(os.path.join(os.path.dirname(cand), '*')):
        shutil.copyfile(f, os.path.join(os.path.dirname(dst), os.path.basename(f)))
    ck = torch.load(dst, map_location='cpu', weights_only=False)
    print(f"RESUMING {cand.split('/runs/')[1].split('/')[0]}: epoch {ck['state']['epoch']}, "
          f"best val CER {ck['state']['best_cer']:.2f}")
    restored += 1
if not restored:
    print('fresh start (no previous run attached as an input)')

In [ ]:
# Cell 2 - configuration
OUT = '/kaggle/working'
WORK = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'     # scratch space, kept out of the saved output
CODE = f'{OUT}/code'
os.makedirs(CODE, exist_ok=True); os.makedirs(f'{OUT}/runs', exist_ok=True)

# Every MoDeTrans patch is exactly 1268 x 463 (checked on all 2,043). Scaled to 320 x 880, each gets
# the same 10 x 110 output grid = 1,100 slots, i.e. about 11 per character for a typical label.
# 320 puts the writing at roughly the scale SPAN sees on IAM at 150 dpi (about 2 rows per text line).
IMG_H, IMG_W = 320, 880

PRETRAINED = dict(url='https://zenodo.org/api/records/7244546/files/Pool-Line-R_iam.pt/content',
                  md5='474a0e196d2194f63ccb7a69ba0ba775')

# one model per GPU
VARIANTS = [dict(tag='scratch',  name='scratch',  init='none', gpu=0),
            dict(tag='iam_init', name='iam_init', init='iam',  gpu=1)]
if N_GPU == 1:
    VARIANTS = [dict(tag='iam_init', name='iam_init', init='iam', gpu=0)]
print('input', IMG_H, 'x', IMG_W, '| variants:', [v['tag'] for v in VARIANTS], '| scratch dir:', WORK)

In [ ]:
# Cell 3 - libraries Kaggle does not ship
!pip -q install datasets editdistance sacrebleu 2>&1 | tail -1
print('installed')

In [ ]:
# Cell 4 - find colab_pack.zip anywhere under /kaggle/input and unpack the metadata
DATA = f'{WORK}/metadata_unzipped'
os.makedirs(DATA, exist_ok=True)
need = ['splits.json', 'modetrans_text.csv', 'charset.json']

def find_metadata():
    for root, _, files in os.walk('/kaggle/input'):
        if all(n in files for n in need):
            return root
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip'):
                zipfile.ZipFile(os.path.join(root, f)).extractall(DATA)
                for r2, _, f2 in os.walk(DATA):
                    if all(n in f2 for n in need):
                        return r2
    return None

META = find_metadata()
assert META, 'Could not find the metadata. Add colab_pack.zip as an input dataset (Add Input -> Upload).'
print('metadata:', META, sorted(os.listdir(META))[:6])

In [ ]:
# Cell 5 - MoDeTrans images + our fixed splits and cleaned text (identical to notebooks 02-03,
# so every number stays comparable with everything measured so far)
import pandas as pd
from datasets import load_dataset

ds = load_dataset('historyHulk/MoDeTrans', split='train')     # 2,043 rows
names  = list(ds['filename'])
splits = json.load(open(f'{META}/splits.json', encoding='utf-8'))['modetrans']['assignment']
text   = (pd.read_csv(f'{META}/modetrans_text.csv', keep_default_na=False)
            .set_index('image_id')['text_train'].to_dict())

assert len(names) == 2043 and set(names) == set(splits) == set(text), 'dataset / split mismatch'
SPLIT_IDS = {s: [n for n in names if splits[n] == s] for s in ['train', 'val', 'test']}
assert not (set(SPLIT_IDS['train']) & (set(SPLIT_IDS['val']) | set(SPLIT_IDS['test']))), 'leakage!'
print('splits:', {k: len(v) for k, v in SPLIT_IDS.items()})

In [ ]:
# Cell 6 - alphabet: index 0 is the CTC blank, then the 86 symbols, then <FRAC> and <unk>
cs = json.load(open(f'{META}/charset.json', encoding='utf-8'))
VOCAB = ['<blank>'] + cs['symbols'] + ['<FRAC>', '<unk>']
STOI = {c: i for i, c in enumerate(VOCAB)}
NUM_CLASSES = len(VOCAB)
_TOK = re.compile(r'<FRAC>|.', re.S)
encode     = lambda s: [STOI.get(t, STOI['<unk>']) for t in _TOK.findall(s)]
decode_ids = lambda ids: ''.join(VOCAB[i] for i in ids if i > 0)

bad = [n for n in names if decode_ids(encode(text[n])) != text[n]]
assert not bad, f'round-trip failed: {bad[:5]}'
assert sum(t == STOI['<unk>'] for n in SPLIT_IDS['train'] for t in encode(text[n])) == 0
print(f'{NUM_CLASSES} classes, longest label {max(len(encode(text[n])) for n in names)} characters')

### Cells 7 and 8 write the two code files
They are written to `/kaggle/working/code/` and are the tested files verbatim, so you can read them
here. `span_lib.py` is the model, loss, augmentation and training loop; `run_span.py` trains one
model on one GPU.

In [ ]:
%%writefile /kaggle/working/code/span_lib.py
"""span_lib.py - SPAN reimplemented for MODI-SYNTH.

SPAN = Simple Predict & Align Network (Coquenet, Chatelain, Paquet; ICDAR 2021, arXiv 2102.08742).
The architecture, loss and training recipe follow the paper and the authors' public code
(github.com/FactoDeepLearning/SPAN, CeCILL-C). This file is a reimplementation, not a copy. Layer
and parameter names are kept identical to theirs on purpose, so that their published weights
(CC-BY-4.0, zenodo.org/records/7244546) load with strict=True - which doubles as a check that this
network really is the same network.

What is taken from them, and where:
  encoder   6 ConvBlocks 1>32>64>128>256>512>512 with strides (1,1)(2,2)(2,2)(2,2)(2,1)(2,1), then
            4 depthwise-separable DSC blocks with residual connections.  paper section 3.1, models.py
  blocks    conv > ReLU > conv > ReLU > InstanceNorm > conv, dropout at ONE of three random places
            per forward pass.                                           paper section 3.1, models.py
  decoder   a single 5x5 convolution to (classes) channels, log-softmax.  paper section 3.2
  loss      CTC on the 2D grid flattened row by row, per sample cropped to its own valid region,
            normalised by label length.                                  paper section 3.2, trainer
  optimiser Adam, lr 1e-4, no scheduler, fp32, dropout 0.5, augmentation 0.2 each.  main_pg_ctc.py
  input     standardised with the training set's own mean and std, AFTER augmentation.
"""
import os, sys, json, math, time, csv, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import cv2

# ------------------------------------------------------------------ geometry
# five stride-2 steps shrink the height (blocks 2-6), three shrink the width (blocks 2-4).
# a 3x3 convolution with padding 1 and stride 2 turns n into ceil(n / 2).
def reduced_shape(h, w):
    for _ in range(5):
        h = (h + 1) // 2
    for _ in range(3):
        w = (w + 1) // 2
    return h, w


# ------------------------------------------------------------------ model
class DepthSepConv2D(nn.Module):
    def __init__(self, cin, cout, stride=(1, 1)):
        super().__init__()
        self.depth_conv = nn.Conv2d(cin, cin, (3, 3), stride=stride, padding=(1, 1), groups=cin)
        self.point_conv = nn.Conv2d(cin, cout, (1, 1))

    def forward(self, x):
        return self.point_conv(self.depth_conv(x))


class MixDropout(nn.Module):
    # either ordinary dropout or whole-channel dropout, chosen at random on every call
    def __init__(self, p=0.4, p2d=0.2):
        super().__init__()
        self.dropout = nn.Dropout(p)
        self.dropout2d = nn.Dropout2d(p2d)

    def forward(self, x):
        return self.dropout(x) if random.random() < 0.5 else self.dropout2d(x)


class ConvBlock(nn.Module):
    def __init__(self, cin, cout, stride=(1, 1), dropout=0.4):
        super().__init__()
        self.activation = nn.ReLU()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.conv3 = nn.Conv2d(cout, cout, 3, padding=1, stride=stride)
        self.norm_layer = nn.InstanceNorm2d(cout, eps=0.001, momentum=0.99, track_running_stats=False)
        self.dropout = MixDropout(dropout, dropout / 2)

    def forward(self, x):
        pos = random.randint(1, 3)                 # dropout goes in exactly one of three places
        x = self.activation(self.conv1(x))
        if pos == 1:
            x = self.dropout(x)
        x = self.activation(self.conv2(x))
        if pos == 2:
            x = self.dropout(x)
        x = self.norm_layer(x)
        x = self.activation(self.conv3(x))
        if pos == 3:
            x = self.dropout(x)
        return x


class DSCBlock(nn.Module):
    def __init__(self, cin, cout, stride=(1, 1), dropout=0.4):
        super().__init__()
        self.activation = nn.ReLU()
        self.conv1 = DepthSepConv2D(cin, cout)
        self.conv2 = DepthSepConv2D(cout, cout)
        self.conv3 = DepthSepConv2D(cout, cout, stride=stride)
        self.norm_layer = nn.InstanceNorm2d(cout, eps=0.001, momentum=0.99, track_running_stats=False)
        self.dropout = MixDropout(dropout, dropout / 2)

    def forward(self, x):
        pos = random.randint(1, 3)
        x = self.activation(self.conv1(x))
        if pos == 1:
            x = self.dropout(x)
        x = self.activation(self.conv2(x))
        if pos == 2:
            x = self.dropout(x)
        x = self.norm_layer(x)
        x = self.conv3(x)                          # no ReLU after the third separable conv
        if pos == 3:
            x = self.dropout(x)
        return x


class FCNEncoder(nn.Module):
    def __init__(self, in_ch=1, dropout=0.5, width_mult=1.0):
        super().__init__()
        ch = [max(2, int(round(c * width_mult))) for c in (32, 64, 128, 256, 512, 512)]
        self.init_blocks = nn.ModuleList([
            ConvBlock(in_ch, ch[0], (1, 1), dropout),
            ConvBlock(ch[0], ch[1], (2, 2), dropout),
            ConvBlock(ch[1], ch[2], (2, 2), dropout),
            ConvBlock(ch[2], ch[3], (2, 2), dropout),
            ConvBlock(ch[3], ch[4], (2, 1), dropout),
            ConvBlock(ch[4], ch[5], (2, 1), dropout),
        ])
        self.blocks = nn.ModuleList([DSCBlock(ch[5], ch[5], (1, 1), dropout) for _ in range(4)])
        self.out_channels = ch[5]

    def forward(self, x):
        for b in self.init_blocks:
            x = b(x)
        for b in self.blocks:
            xt = b(x)
            x = x + xt if x.size() == xt.size() else xt
        return x


class Decoder(nn.Module):
    def __init__(self, cin, n_classes):
        super().__init__()
        self.end_conv = nn.Conv2d(cin, n_classes, kernel_size=(5, 5), padding=(2, 2))

    def forward(self, x):
        return F.log_softmax(self.end_conv(x), dim=1)


class SPAN(nn.Module):
    def __init__(self, n_classes, in_ch=1, dropout=0.5, width_mult=1.0):
        super().__init__()
        self.encoder = FCNEncoder(in_ch, dropout, width_mult)
        self.decoder = Decoder(self.encoder.out_channels, n_classes)

    def forward(self, x):
        return self.decoder(self.encoder(x))       # (B, classes, rows, cols) log-probabilities


def load_pretrained_encoder(model, path, log=print):
    """Initialise the encoder from the authors' IAM weights.

    Their network reads 3-channel images (a grey image copied three times). A 1-channel network
    gives the identical result if the first convolution's weights are summed over the input
    channels, so that is what is done: it is exact, not an approximation.
    """
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = None
    for key in ("encoder_state_dict", "encoder", "model_state_dict"):
        if isinstance(ck, dict) and key in ck:
            sd = ck[key]
            break
    if sd is None and isinstance(ck, dict) and all(torch.is_tensor(v) for v in ck.values()):
        sd = ck
    assert sd is not None, f"no encoder weights found; checkpoint keys: {list(ck)[:12]}"
    sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
    w = sd["init_blocks.0.conv1.weight"]
    if w.shape[1] == 3 and model.encoder.init_blocks[0].conv1.weight.shape[1] == 1:
        sd["init_blocks.0.conv1.weight"] = w.sum(dim=1, keepdim=True)
    missing, unexpected = model.encoder.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"encoder mismatch: missing {missing[:4]} unexpected {unexpected[:4]}"
    log(f"loaded IAM-pretrained encoder from {os.path.basename(path)} "
        f"({sum(p.numel() for p in model.encoder.parameters())/1e6:.1f} M parameters, strict match)")


# ------------------------------------------------------------------ loss and decoding
def ctc_loss_spatial(logp, targets, rshapes):
    """CTC on the 2D grid. Each sample is cut to its own valid (rows, cols) region, flattened row by
    row, and its loss is divided by its own label length (SPAN's 'mean' reduction). Returns the mean
    over the batch and the per-sample values."""
    B, C = logp.shape[0], logp.shape[1]
    seqs, lens = [], []
    for i in range(B):
        rh, rw = rshapes[i]
        seqs.append(logp[i, :, :rh, :rw].reshape(C, rh * rw).t())      # (T_i, C)
        lens.append(rh * rw)
    padded = pad_sequence(seqs)                                         # (T_max, B, C)
    tl = torch.tensor([len(t) for t in targets], dtype=torch.long)
    il = torch.tensor(lens, dtype=torch.long)
    per = F.ctc_loss(padded.float(), torch.cat(targets).to(padded.device), il, tl,
                     blank=0, reduction="none", zero_infinity=True)
    return (per / tl.to(per.device).clamp(min=1)).mean(), per


def greedy_decode(logp, rshapes, decode_ids):
    ids = logp.argmax(1).cpu().numpy()
    hyps, blank_hit, blank_tot = [], 0, 0
    for i, (rh, rw) in enumerate(rshapes):
        s = ids[i, :rh, :rw].reshape(-1)
        blank_hit += int((s == 0).sum())
        blank_tot += s.size
        keep = np.ones_like(s, dtype=bool)
        keep[1:] = s[1:] != s[:-1]
        keep &= s != 0
        hyps.append(decode_ids(s[keep]))
    return hyps, blank_hit / max(1, blank_tot)


# ------------------------------------------------------------------ metrics (same as notebooks 01-03)
def compute_metrics(refs, hyps, n_boot=0, seed=0):
    import editdistance, sacrebleu
    u = lambda s: s.replace("<FRAC>", "")
    e = np.array([editdistance.eval(u(r), u(h)) for r, h in zip(refs, hyps)], float)
    L = np.array([max(1, len(u(r))) for r in refs], float)
    we = np.array([editdistance.eval(r.split(), h.split()) for r, h in zip(refs, hyps)], float)
    wl = np.array([max(1, len(r.split())) for r in refs], float)
    per = e / L
    out = dict(CER=100 * e.sum() / L.sum(), WER=100 * we.sum() / wl.sum(),
               median_CER=100 * float(np.median(per)), catastrophic_pct=100 * float((per > 0.5).mean()),
               chrF_pp=sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score,
               BLEU=sacrebleu.corpus_bleu(hyps, [refs]).score, n=len(refs))
    if n_boot:
        rg = np.random.default_rng(seed)
        idx = rg.integers(0, len(refs), (n_boot, len(refs)))
        b = 100 * e[idx].sum(1) / L[idx].sum(1)
        out["CER_CI95"] = (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))
    return out


# ------------------------------------------------------------------ augmentation (SPAN: each at p=0.2)
def augment(img, rng, p=0.2):
    h, w = img.shape
    if rng.random() < p:                                    # resolution: scale by 0.75-1.0
        f = rng.uniform(0.75, 1.0)
        img = cv2.resize(img, (max(8, int(math.ceil(w * f))), max(8, int(math.ceil(h * f)))),
                         interpolation=cv2.INTER_LINEAR)
        h, w = img.shape
    if rng.random() < p:                                    # perspective, distortion scale 0-0.3
        s = rng.uniform(0.0, 0.3)
        dx, dy = int(s * (w // 2)), int(s * (h // 2))
        dst = np.float32([[rng.integers(0, dx + 1), rng.integers(0, dy + 1)],
                          [rng.integers(w - dx - 1, w), rng.integers(0, dy + 1)],
                          [rng.integers(w - dx - 1, w), rng.integers(h - dy - 1, h)],
                          [rng.integers(0, dx + 1), rng.integers(h - dy - 1, h)]])
        src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
        img = cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (w, h),
                                  flags=cv2.INTER_LINEAR, borderValue=255)
    elif rng.random() < p:                                  # elastic distortion along one axis
        mag, k = int(rng.integers(1, 21)), int(rng.integers(1, 4))
        mx, my = (mag, 1) if rng.integers(2) == 0 else (1, mag)
        gx = rng.uniform(-mx, mx, (k + 1, k + 1)).astype(np.float32)
        gy = rng.uniform(-my, my, (k + 1, k + 1)).astype(np.float32)
        fx = cv2.resize(gx, (w, h), interpolation=cv2.INTER_CUBIC)
        fy = cv2.resize(gy, (w, h), interpolation=cv2.INTER_CUBIC)
        mapx, mapy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
        img = cv2.remap(img, mapx + fx, mapy + fy, cv2.INTER_LINEAR, borderValue=255)
    elif rng.random() < p:                                  # random projective transform, one axis
        dw, dh = (0.10 * w, 0.0) if rng.integers(2) == 0 else (0.0, 0.15 * h)
        sh = lambda d: rng.uniform(-d, d)
        dst = np.float32([[sh(dw), sh(dh)], [w - 1 - sh(dw), sh(dh)],
                          [w - 1 - sh(dw), h - 1 - sh(dh)], [sh(dw), h - 1 - sh(dh)]])
        src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
        img = cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (w, h),
                                  flags=cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < p:                                    # dilation or erosion, kernel 1-3
        k = np.ones((int(rng.integers(1, 4)), int(rng.integers(1, 4))), np.uint8)
        img = cv2.erode(img, k) if rng.integers(2) == 0 else cv2.dilate(img, k)
    # contrast and brightness: SPAN draws the factor from 0.01-1. Measured on our patches that leaves
    # about 3% of augmented pages with under 10% of their original contrast - effectively blank, yet
    # still carrying a full label, which teaches the model to invent text. The floor is raised to
    # 0.3. This is the ONE deliberate deviation from SPAN's augmentation, and it comes from the
    # image audit (scripts/20_span_input_audit.py), not from taste.
    if rng.random() < p:                                    # contrast (blend with the mean)
        f, m = rng.uniform(0.3, 1.0), float(img.mean())
        img = np.clip(m + f * (img.astype(np.float32) - m), 0, 255).astype(np.uint8)
    if rng.random() < p:                                    # brightness
        img = np.clip(img.astype(np.float32) * rng.uniform(0.3, 1.0), 0, 255).astype(np.uint8)
    if rng.random() < p:                                    # sign flipping
        img = 255 - img
    return img


# ------------------------------------------------------------------ data
class Pages(torch.utils.data.Dataset):
    def __init__(self, data, ids, train, aug_p):
        self.d, self.ids, self.train, self.aug_p = data, list(ids), train, aug_p

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        name = self.ids[i]
        img = np.array(self.d["imgs"][self.d["idx"][name]])            # (H, W) uint8, copied out of the memmap
        if self.train and self.aug_p > 0:
            seed = int(torch.randint(0, 2 ** 31 - 1, (1,)).item())
            img = augment(img, np.random.default_rng(seed), self.aug_p)
        lab = self.d["labels"][name]
        rh, rw = reduced_shape(*img.shape)
        need = 2 * len(lab) + 1                                        # SPAN's 'CTC_pg' constraint
        if need > rh * rw:
            new_w = int(math.ceil(need / rh)) * 8
            if new_w > img.shape[1]:
                img = np.pad(img, ((0, 0), (0, new_w - img.shape[1])), constant_values=255)
            rh, rw = reduced_shape(*img.shape)
        x = (img.astype(np.float32) - self.d["mean"]) / self.d["std"]   # standardise AFTER augmentation
        return torch.from_numpy(x)[None], torch.as_tensor(lab, dtype=torch.long), (rh, rw), name


def collate(batch):
    xs, ys, rs, ns = zip(*batch)
    H = max(x.shape[1] for x in xs)
    W = max(x.shape[2] for x in xs)
    out = torch.zeros(len(xs), 1, H, W)             # 0 is the mean intensity once standardised: SPAN's padding value
    for i, x in enumerate(xs):
        out[i, :, :x.shape[1], :x.shape[2]] = x
    return out, list(ys), list(rs), list(ns)


def load_data(path):
    meta = json.load(open(path, encoding="utf-8"))
    d = dict(meta)
    d["imgs"] = np.load(meta["npy"], mmap_mode="r")
    d["idx"] = {n: i for i, n in enumerate(meta["names"])}
    vocab = meta["vocab"]
    d["decode_ids"] = lambda ids: "".join(vocab[i] for i in ids if i > 0)
    return d


def make_loader(data, ids, train, batch_size, aug_p, workers, pin):
    return torch.utils.data.DataLoader(
        Pages(data, ids, train, aug_p), batch_size=batch_size, shuffle=train,
        drop_last=bool(train) and len(ids) > batch_size, num_workers=workers, collate_fn=collate,
        pin_memory=pin, persistent_workers=workers > 0)


@torch.no_grad()
def evaluate(model, loader, data, device):
    model.eval()
    refs, hyps, names, bl, nb = [], [], [], 0.0, 0
    for x, ys, rs, ns in loader:
        logp = model(x.to(device))
        h, b = greedy_decode(logp, rs, data["decode_ids"])
        hyps += h
        refs += [data["text"][n] for n in ns]
        names += ns
        bl += b
        nb += 1
    return refs, hyps, names, bl / max(1, nb)


# ------------------------------------------------------------------ training
def _save(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)                 # never leave a half-written checkpoint behind


def fit(cfg, data, run_dir, train_ids, val_ids, device, init="none", pretrained_path=None,
        deadline=None, stop_at_cer=None, verbose_every=1, log=print):
    os.makedirs(run_dir, exist_ok=True)
    random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    n_classes = len(data["vocab"])
    model = SPAN(n_classes, 1, cfg["dropout"], cfg.get("width_mult", 1.0))
    opt_state, state = None, dict(epoch=0, best_cer=1e9, best_epoch=0, bad_epochs=0, sessions=0)
    last = f"{run_dir}/last.pt"
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); opt_state = ck["opt"]; state = ck["state"]
        state["sessions"] = state.get("sessions", 0) + 1
        log(f"RESUMED at epoch {state['epoch'] + 1} (session #{state['sessions'] + 1}); "
            f"best val CER so far {state['best_cer']:.2f} at epoch {state['best_epoch']}")
    elif init == "iam":
        load_pretrained_encoder(model, pretrained_path, log)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], amsgrad=False)
    if opt_state is not None:
        opt.load_state_dict(opt_state)
    json.dump(dict(cfg, init=init), open(f"{run_dir}/config.json", "w"), indent=2)

    pin = str(device).startswith("cuda")
    tl = make_loader(data, train_ids, True, cfg["batch_size"], cfg["aug_p"] if cfg["use_aug"] else 0.0,
                     cfg["num_workers"], pin)
    vl = make_loader(data, val_ids, False, cfg["batch_size"], 0.0, cfg["num_workers"], pin)
    logf = f"{run_dir}/log.csv"
    COLS = ["epoch", "train_loss", "val_CER", "val_WER", "val_median_CER", "val_catastrophic_pct",
            "val_chrF_pp", "blank_slot_pct", "pred_len", "gt_len", "grad_norm", "weight_norm",
            "skipped", "sec", "elapsed_h"]
    if not os.path.exists(logf):
        csv.writer(open(logf, "w", newline="")).writerow(COLS)
    gt_len = float(np.mean([len(data["text"][n]) for n in val_ids]))
    t_start = time.time(); last_sec = 0.0

    while state["epoch"] < cfg["max_epochs"] and state["bad_epochs"] < cfg["patience"]:
        if deadline is not None and time.time() + max(60.0, 1.15 * last_sec) > deadline:
            log(f"TIME BUDGET reached at epoch {state['epoch']}; last.pt is saved. Attach this "
                f"notebook's output as an input and commit again to continue at epoch {state['epoch'] + 1}.")
            break
        t0 = time.time(); model.train(); tot = nb = skipped = 0; gmax = 0.0
        for x, ys, rs, ns in tl:
            logp = model(x.to(device, non_blocking=True))
            loss, _ = ctc_loss_spatial(logp, [y.to(device) for y in ys], rs)
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); skipped += 1; continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            # SPAN does not clip gradients. The limit here is so high it only catches a genuine
            # explosion, and a non-finite gradient (loss finite, gradient not) skips the step.
            g = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.get("grad_clip", 1000.0)))
            if not math.isfinite(g):
                opt.zero_grad(set_to_none=True); skipped += 1; continue
            gmax = max(gmax, g)
            opt.step()
            tot += loss.item(); nb += 1
        train_loss = tot / max(1, nb)
        wnorm = float(sum(p.detach().float().pow(2).sum() for p in model.parameters()).sqrt())
        if skipped > 0.05 * len(tl):
            log(f"ABORTED: {skipped}/{len(tl)} batches gave a non-finite loss (weight norm {wnorm:,.0f}).")
            break
        refs, hyps, _, blank = evaluate(model, vl, data, device)
        m = compute_metrics(refs, hyps)
        state["epoch"] += 1
        improved = m["CER"] < state["best_cer"] - cfg["min_delta"]
        if improved:
            state.update(best_cer=m["CER"], best_epoch=state["epoch"], bad_epochs=0)
            _save({"model": model.state_dict(), "cfg": cfg, "epoch": state["epoch"], "val": m},
                  f"{run_dir}/best.pt")
        else:
            state["bad_epochs"] += 1
        _save({"model": model.state_dict(), "opt": opt.state_dict(), "state": state}, last)
        sec = time.time() - t0; last_sec = sec
        pred_len = float(np.mean([len(h) for h in hyps]))
        csv.writer(open(logf, "a", newline="")).writerow(
            [state["epoch"], round(train_loss, 4), round(m["CER"], 2), round(m["WER"], 2),
             round(m["median_CER"], 2), round(m["catastrophic_pct"], 1), round(m["chrF_pp"], 2),
             round(100 * blank, 1), round(pred_len, 1), round(gt_len, 1), round(gmax, 2),
             round(wnorm, 1), skipped, round(sec), round((time.time() - t_start) / 3600, 3)])
        if state["epoch"] % verbose_every == 0 or improved:
            left = ""
            if deadline is not None:
                left = f" | ~{max(0, (deadline - time.time()) / max(sec, 1)):.0f} epochs left this session"
            tag = "  *BEST*" if improved else f"  ({state['bad_epochs']}/{cfg['patience']} stale)"
            log(f"ep {state['epoch']:4d} | loss {train_loss:6.3f} | CER {m['CER']:6.2f} "
                f"(med {m['median_CER']:5.1f}) | WER {m['WER']:6.2f} | chrF {m['chrF_pp']:5.1f} | "
                f"best {state['best_cer']:6.2f}@{state['best_epoch']:<4d} | blank {100 * blank:4.1f}% | "
                f"len {pred_len:5.1f}/{gt_len:.0f} | g {gmax:5.1f} w {wnorm:6.0f} | {sec:4.0f}s{left}{tag}")
            log(f"   GT  : {refs[0][:85]}")
            log(f"   PRED: {hyps[0][:85]}")
        if stop_at_cer is not None and m["CER"] <= stop_at_cer:
            log(f"reached CER {m['CER']:.2f}, stopping"); break
    log(f"done: {state['epoch']} epochs, best val CER {state['best_cer']:.2f} at epoch {state['best_epoch']}")
    return model, state


# ------------------------------------------------------------------ data preparation
def build_cache(pil_images, n, H, W, path):
    """Scale every patch to H x W once and keep them all in one uint8 array on disk. Nothing is cut,
    split, reordered or discarded: this is the entire image pipeline."""
    arr = np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8, shape=(n, H, W))
    for i, pil in enumerate(pil_images):
        arr[i] = cv2.resize(np.array(pil.convert("L")), (W, H), interpolation=cv2.INTER_AREA)
    arr.flush()
    return arr


def train_stats(arr, rows):
    """Mean and standard deviation of the TRAINING images only, as SPAN standardises its input."""
    tot = tot2 = 0.0
    for r in rows:
        a = np.asarray(arr[r], dtype=np.float64)
        tot += a.sum(); tot2 += (a * a).sum()
    n = len(rows) * arr.shape[1] * arr.shape[2]
    mean = tot / n
    return float(mean), float(math.sqrt(max(tot2 / n - mean * mean, 1e-12)))


# ------------------------------------------------------------------ one process per GPU
def launch_jobs(jobs, script, data_json, out, ckpt, deadline, extra_args=(), poll=15.0, log=print,
                stop_when=None):
    """Start one training process per job, each pinned to its own GPU, and stream their output.

    Separate processes (not threads, not DataParallel) so that two different models can train at the
    same time on the two T4s a Kaggle session provides. Each writes to its own run directory, so a
    crash in one cannot touch the other. If stop_when() becomes true, the remaining processes are
    ended (the smoke test uses this: once one model has proved the pipeline, waiting for the slower
    one only leaves a GPU idle).
    """
    import subprocess
    os.makedirs(f"{out}/runs", exist_ok=True)
    procs = []
    for j in jobs:
        path = f"{out}/runs/{j['tag']}.stdout"
        fh = open(path, "w", encoding="utf-8")
        cmd = [sys.executable, "-u", script, "--data", data_json, "--out", out, "--name", j["name"],
               "--init", j["init"], "--deadline", str(deadline)]
        cmd += (["--ckpt", ckpt] if ckpt else []) + (["--smoke"] if j.get("smoke") else []) + list(extra_args)
        env = dict(os.environ)
        if j.get("gpu") is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(j["gpu"])
        procs.append(dict(job=j, p=subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env),
                          path=path, fh=fh, pos=0))

    def drain(P):
        with open(P["path"], "r", encoding="utf-8", errors="replace") as f:
            f.seek(P["pos"]); chunk = f.read(); P["pos"] = f.tell()
        for line in chunk.splitlines():
            log(f"[{P['job']['tag']}] {line}")

    while any(P["p"].poll() is None for P in procs):
        time.sleep(poll)
        for P in procs:
            drain(P)
        if stop_when is not None and stop_when():
            log("stop condition met - ending the processes that are still running")
            for P in procs:
                if P["p"].poll() is None:
                    P["p"].terminate()
            time.sleep(5)
            for P in procs:
                if P["p"].poll() is None:
                    P["p"].kill()
            break
        if time.time() > deadline + 900:                    # children obey the deadline; this is the backstop
            for P in procs:
                if P["p"].poll() is None:
                    P["p"].kill()
    for P in procs:
        drain(P); P["fh"].close()
    return {P["job"]["tag"]: P["p"].returncode for P in procs}


In [ ]:
%%writefile /kaggle/working/code/run_span.py
"""run_span.py - train ONE SPAN variant in its own process, on the GPU it is pinned to.

The notebook starts one of these per GPU, so a from-scratch model and an IAM-initialised model can
train side by side in the same session and be compared on equal terms.
"""
import argparse, json, os, sys, csv

ap = argparse.ArgumentParser()
ap.add_argument("--data", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--name", required=True)
ap.add_argument("--init", default="none", choices=["none", "iam"])
ap.add_argument("--ckpt", default="")
ap.add_argument("--gpu", default=None)
ap.add_argument("--device", default="cuda")
ap.add_argument("--deadline", type=float, default=None)
ap.add_argument("--smoke", action="store_true")
ap.add_argument("--smoke-n", type=int, default=20)
ap.add_argument("--width-mult", type=float, default=1.0)      # 1.0 is the real network; smaller is for plumbing tests only
ap.add_argument("--max-epochs", type=int, default=None)
ap.add_argument("--workers", type=int, default=2)
ap.add_argument("--limit-train", type=int, default=None)      # plumbing tests only
ap.add_argument("--limit-val", type=int, default=None)
args = ap.parse_args()
if args.gpu is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu              # must be set before torch is imported
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import span_lib as L

data = L.load_data(args.data)
cfg = dict(seed=42, batch_size=4, lr=1e-4, dropout=0.5, aug_p=0.2, use_aug=True, max_epochs=5000,
           patience=500, min_delta=0.05, num_workers=args.workers, grad_clip=1000.0,
           width_mult=args.width_mult)
if args.smoke:
    # memorise a handful of patches. lr 1e-3 and dropout 0.1 are deliberate: a memorisation test
    # wants updates, not regularisation (the first Kaggle smoke test inherited the main learning
    # rate by accident, took 140 epochs to leave the prior, and failed a healthy pipeline).
    ids = data["splits"]["train"][:args.smoke_n]
    cfg.update(lr=1e-3, batch_size=2, dropout=0.1, use_aug=False, max_epochs=500, patience=500)
    train_ids, val_ids, stop_at = ids, ids, 5.0
    run_dir = f"{args.out}/runs/smoke_{args.name}"
else:
    train_ids = data["splits"]["train"][:args.limit_train]
    val_ids = data["splits"]["val"][:args.limit_val or 150]
    stop_at = None
    run_dir = f"{args.out}/runs/{args.name}"
if args.max_epochs:
    cfg["max_epochs"] = args.max_epochs

print(f"init={args.init} device={args.device} train={len(train_ids)} val={len(val_ids)} "
      f"batch={cfg['batch_size']} lr={cfg['lr']} dropout={cfg['dropout']} aug={cfg['use_aug']}", flush=True)
model, state = L.fit(cfg, data, run_dir, train_ids, val_ids, args.device, init=args.init,
                     pretrained_path=args.ckpt or None, deadline=args.deadline, stop_at_cer=stop_at,
                     log=lambda s: print(s, flush=True))

if args.smoke:
    rows = list(csv.DictReader(open(f"{run_dir}/log.csv")))
    best = min(rows, key=lambda r: float(r["val_CER"]))
    res = dict(variant=args.name, epochs=len(rows), best_epoch=int(best["epoch"]),
               best_CER=float(best["val_CER"]), loss_at_best=float(best["train_loss"]),
               blank_at_best=float(best["blank_slot_pct"]))
    res["passed"] = res["best_CER"] <= 10.0 and res["loss_at_best"] <= 2.5 and res["blank_at_best"] <= 90.0
    json.dump(res, open(f"{run_dir}/result.json", "w"), indent=2)
    print(f"SMOKE RESULT {json.dumps(res)}", flush=True)


In [ ]:
# Cell 9 - images. THE WHOLE PATCH, scaled once to 320 x 880. Nothing is cut, split, deskewed,
# reordered or thrown away. This is the entire image pipeline.
#
# The earlier notebooks found the text lines first and laid them end to end. That needed a line
# finder, and it failed three separate ways (lines merged and crushed into one band, header lines
# dropped, ink discarded) before any of them could be seen. SPAN takes the page as it is and lets the
# network learn the alignment, so that whole class of failure cannot occur.
import numpy as np
sys.path.insert(0, CODE)
import span_lib as L

NPY = f'{WORK}/pages_{IMG_H}x{IMG_W}.npy'
if not os.path.exists(NPY):
    L.build_cache((r['image'] for r in ds), len(names), IMG_H, IMG_W, NPY)
arr = np.load(NPY, mmap_mode='r')
IDX = {n: i for i, n in enumerate(names)}
mean, std = L.train_stats(arr, [IDX[n] for n in SPLIT_IDS['train']])
LABELS = {n: encode(text[n]) for n in names}
DATA_JSON = f'{WORK}/data.json'
json.dump(dict(npy=NPY, names=names, splits=SPLIT_IDS, labels=LABELS, text={n: text[n] for n in names},
               vocab=VOCAB, mean=mean, std=std), open(DATA_JSON, 'w', encoding='utf-8'), ensure_ascii=False)

# CTC needs a slot for every character. SPAN requires at least 2*len+1 slots (room for any repeats),
# and a sample that fails this returns ZERO loss silently, so check every patch.
rh, rw = L.reduced_shape(IMG_H, IMG_W)
lab = np.array([len(LABELS[n]) for n in names])
print(f'cached {arr.shape}  |  train mean {mean:.1f} std {std:.1f}')
print(f'output grid {rh} rows x {rw} cols = {rh*rw} slots | labels {lab.min()}-{lab.max()} characters')
print(f'slots per character: median {np.median(rh*rw/lab):.1f}, worst {(rh*rw/lab).min():.1f}')
assert 2 * lab.max() + 1 <= rh * rw, 'some label is too long for the grid'
print(f"every patch satisfies SPAN's rule (>= 2*len+1 = {2*lab.max()+1} slots needed at most, {rh*rw} available)")

In [ ]:
# Cell 10 - VISUAL AUDIT, saved to /kaggle/working/image-processing-audit/. Far shorter than the
# line-cutting audit of notebook 03, because the only edits left are one scale and the augmentations.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
AUD = f'{OUT}/image-processing-audit'; os.makedirs(AUD, exist_ok=True)

rs = np.random.default_rng(42)
for n in [names[i] for i in rs.choice(len(names), 4, replace=False)]:
    j = IDX[n]; img = np.array(arr[j])
    fig, ax = plt.subplots(3, 2, figsize=(16, 9)); ax = ax.ravel()
    ax[0].imshow(img, cmap='gray', vmin=0, vmax=255)
    for r in range(1, rh): ax[0].axhline(r * IMG_H / rh - 0.5, color='tab:red', lw=0.5, alpha=0.6)
    ax[0].set_title(f'{n}: the model input. red = the {rh} rows of the output grid; '
                    f'{rh*rw} slots for {len(LABELS[n])} characters = {rh*rw/len(LABELS[n]):.1f} per character', fontsize=8)
    ax[0].axis('off')
    for k in range(1, 6):
        a = L.augment(img.copy(), np.random.default_rng(100 + k), 0.2)
        ax[k].imshow(a, cmap='gray', vmin=0, vmax=255)
        ax[k].set_title(f'augmented sample {k} (SPAN rate p=0.2), {a.shape[1]}x{a.shape[0]}', fontsize=8); ax[k].axis('off')
    fig.tight_layout(); fig.savefig(f'{AUD}/{n.replace(".jpg","")}_input_and_augmented.png', dpi=80); plt.close(fig)

# how often does an augmentation leave a page nearly unreadable? measured, not assumed
rng = np.random.default_rng(0); ratios = []
for n in [names[i] for i in rs.choice(len(names), 40, replace=False)]:
    img = np.array(arr[IDX[n]]); base = float(img.std())
    ratios += [float(L.augment(img.copy(), rng, 0.2).std()) / max(base, 1e-6) for _ in range(10)]
ratios = np.array(ratios)
print(f'augmented pages with contrast below 25% of the original: {100*(ratios<0.25).mean():.1f}%   '
      f'below 10%: {100*(ratios<0.10).mean():.1f}%   (SPAN\'s 0.01 floor gave ~3% below 10%)')
print(f'wrote {len(os.listdir(AUD))} images to {AUD}')

In [ ]:
# Cell 11 - the IAM-trained encoder from the SPAN authors (CC-BY-4.0). The file is checked against the
# publisher's checksum, then loaded into our network with strict=True, which fails loudly unless every
# layer name and shape matches - a check that this network really is the same network.
CKPT = f'{WORK}/Pool-Line-R_iam.pt'
def md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

HAVE_PRETRAINED = False
for attempt in range(1, 4):
    try:
        if not (os.path.exists(CKPT) and md5(CKPT) == PRETRAINED['md5']):
            print(f'downloading the pre-trained encoder (218 MB), attempt {attempt}')
            subprocess.run(['curl', '-sL', '--fail', '-o', CKPT, PRETRAINED['url']], check=True, timeout=1200)
        if md5(CKPT) == PRETRAINED['md5']:
            HAVE_PRETRAINED = True; break
        print('checksum mismatch, retrying')
    except Exception as e:
        print('download problem:', e)

if HAVE_PRETRAINED:
    _m = L.SPAN(NUM_CLASSES, 1, 0.5)
    L.load_pretrained_encoder(_m, CKPT)                 # raises unless every name and shape matches
    del _m
    print('checksum OK, architecture matches the authors\' trained network')
else:
    print('WARNING: could not get the pre-trained weights. Only the from-scratch model will run.')
    VARIANTS = [v for v in VARIANTS if v['init'] == 'none'] or VARIANTS[:1]
print('will train:', [v['tag'] for v in VARIANTS])

In [ ]:
# Cell 12 - SAFEGUARD: can each model memorise 20 patches? Both run at once, one per GPU.
# This proves images, labels, alphabet, CTC lengths, decoding and metrics are wired up correctly
# before the long run starts. It also gives the first honest comparison between the two models:
# the IAM-initialised one should get there noticeably sooner.
#
# lr 1e-3 and dropout 0.1 here are deliberate: a memorisation test wants updates, not regularisation.
# (An earlier smoke test silently inherited the main learning rate, took 140 epochs just to leave the
# prior, and failed a healthy pipeline.) A model passes at CER <= 10 with loss <= 2.5 and <= 90% blanks.
def smoke_result(v):
    p = f'{OUT}/runs/smoke_{v["name"]}/result.json'
    return json.load(open(p)) if os.path.exists(p) else None

def any_smoke_passed():
    return any((smoke_result(v) or {}).get('passed') for v in VARIANTS)

# a model that already has a smoke result (from an earlier session) is not run again. And as soon as ONE
# model has passed, the pipeline is proven, so the other is ended rather than leaving a GPU idle.
todo = [dict(v, smoke=True) for v in VARIANTS if smoke_result(v) is None]
if todo and not any_smoke_passed():
    rc = L.launch_jobs(todo, f'{CODE}/run_span.py', DATA_JSON, OUT, CKPT if HAVE_PRETRAINED else '',
                       min(DEADLINE, time.time() + 75 * 60), poll=20, stop_when=any_smoke_passed)
    print('exit codes:', rc)
else:
    print('smoke test result already on file from an earlier session - skipping')

print()
results = {v['tag']: smoke_result(v) for v in VARIANTS}
for tag, r in results.items():
    print(f'{tag:9s}', 'no result (crashed?)' if r is None else
          f"best CER {r['best_CER']:6.2f} at epoch {r['best_epoch']:3d} | loss {r['loss_at_best']:.2f} | "
          f"blank {r['blank_at_best']:.1f}% | {'PASSED' if r['passed'] else 'did not pass'}")
passed = [t for t, r in results.items() if r and r['passed']]
assert passed, ('SMOKE TEST FAILED for every model. A model that is not reading scores about 84 CER and sits '
                'at loss 3.3 with >90% blanks. Check runs/smoke_*/log.csv: if CER was still falling steeply it '
                'needed more epochs; if it never left the prior, something in the pipeline IS broken.')
print(f'\nsmoke test passed for {passed}. Starting the long run.')

In [ ]:
# Cell 13 - the long run. One process per GPU, streamed below with [scratch] / [iam_init] prefixes.
#
# WHAT TO EXPECT, so nobody kills a healthy run again:
#   * the first epochs sit near loss 3.3 (the letter-frequency prior) with blank near 100%. Normal.
#   * the paper's own curves fall smoothly from the first epochs and reach loss ~1.2 around epoch 100
#     on data of our size, so judge progress over tens to hundreds of epochs, not a handful.
#   * the honest signs of life: loss below ~3.0, blank falling, len climbing toward the true length.
#   * the decision point is about epoch 20: compare the two models' loss. If NEITHER is below ~3.0 by
#     epoch 40, stop and send the log rather than waiting out the session.
# Each model stops by itself at the time budget and resumes next session from last.pt.
print(f'training {[v["tag"] for v in VARIANTS]} until {BUDGET_H} h; '
      f'{len(SPLIT_IDS["train"])//4} updates per epoch (batch 4)\n')
rc = L.launch_jobs(VARIANTS, f'{CODE}/run_span.py', DATA_JSON, OUT, CKPT if HAVE_PRETRAINED else '',
                   DEADLINE, poll=30)
print('\nexit codes:', rc)

In [ ]:
# Cell 14 - final score on the held-out test split for each model, using its best checkpoint
data = L.load_data(DATA_JSON)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
FINAL = {}
for v in VARIANTS:
    rd = f'{OUT}/runs/{v["name"]}'
    if not os.path.exists(f'{rd}/best.pt'):
        print(f'{v["tag"]}: no checkpoint yet'); continue
    best = torch.load(f'{rd}/best.pt', map_location=dev, weights_only=False)
    wm = json.load(open(f'{rd}/config.json')).get('width_mult', 1.0)
    model = L.SPAN(NUM_CLASSES, 1, 0.5, wm).to(dev); model.load_state_dict(best['model'])
    tl = L.make_loader(data, SPLIT_IDS['test'], False, 4, 0.0, 2 if dev == 'cuda' else 0, dev == 'cuda')
    refs, hyps, ns, blank = L.evaluate(model, tl, data, dev)
    res = L.compute_metrics(refs, hyps, n_boot=10000)
    res.update(variant=v['tag'], best_epoch=best['epoch'], val_CER=best['val']['CER'], blank_slot_pct=round(100*blank, 1))
    json.dump(res, open(f'{OUT}/test_metrics_{v["tag"]}.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
    pd.DataFrame({'image_id': ns, 'reference': refs, 'prediction': hyps}).to_csv(
        f'{OUT}/test_predictions_{v["tag"]}.csv', index=False, encoding='utf-8')
    FINAL[v['tag']] = res
    print(f"{v['tag']:9s} TEST CER {res['CER']:6.2f}  (95% CI {res['CER_CI95'][0]:.1f}-{res['CER_CI95'][1]:.1f}) | "
          f"WER {res['WER']:6.2f} | chrF++ {res['chrF_pp']:5.1f} | best epoch {res['best_epoch']} (val CER {res['val_CER']:.2f})")
if len(FINAL) == 2:
    a, b = FINAL['scratch']['CER'], FINAL['iam_init']['CER']
    print(f"\nIAM-initialised vs from scratch: {b:.2f} vs {a:.2f} CER ({'better' if b < a else 'worse'} by {abs(a-b):.2f} points)")

In [ ]:
# Cell 15 - curves for the report: both models on the same axes
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(19, 4.5))
for v in VARIANTS:
    p = f'{OUT}/runs/{v["name"]}/log.csv'
    if not os.path.exists(p): continue
    d = pd.read_csv(p)
    ax[0].plot(d.epoch, d.train_loss, label=v['tag']); ax[1].plot(d.epoch, d.val_CER, label=v['tag'])
    ax[2].plot(d.epoch, d.blank_slot_pct, label=v['tag'] + ' blank %')
ax[0].axhline(3.32, color='r', ls=':', label='letter-frequency prior'); ax[0].set_title('training loss'); ax[0].legend()
ax[1].axhline(84, color='r', ls=':', label='no-reading baseline'); ax[1].set_title('validation CER'); ax[1].legend()
ax[2].set_title('share of output slots that are blank'); ax[2].legend()
for a in ax: a.set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{OUT}/curves.png', dpi=120); plt.show()
for v in VARIANTS:
    p = f'{OUT}/runs/{v["name"]}/log.csv'
    if os.path.exists(p):
        d = pd.read_csv(p); print(f"{v['tag']:9s} {len(d)} epochs, {d.sec.tail(10).mean():.0f} s/epoch, "
                                  f"best val CER {d.val_CER.min():.2f} at epoch {int(d.loc[d.val_CER.idxmin(), 'epoch'])}")

## Reading the output

| column in `runs/<model>/log.csv` | what it tells you |
|---|---|
| `train_loss` | CTC loss per character. 3.32 is the letter-frequency prior — a model using no image information. Below 3.0 it has started using the picture. |
| `val_CER` | the headline number, micro-averaged over characters, on the first 150 validation patches. |
| `blank_slot_pct` | share of output slots predicting `<blank>`. Near 100% means it is emitting nothing yet. Watch this before CER. |
| `pred_len` / `gt_len` | predicted vs true character count. Climbing toward parity is the clearest early sign of life. |
| `val_chrF_pp` | chrF++ rewards partially correct words and rises before CER falls. |
| `grad_norm` / `weight_norm` | stability. A weight norm that doubles within a few epochs preceded both earlier blow-ups. |
| `sec` | seconds per epoch, which tells you how many epochs one session buys. |

**Baselines.** Random text of the correct length scores about 84 CER; the right letters in the wrong
order also about 84. Anything clearly below that is real signal.

**After the run.** Send `runs/scratch/log.csv`, `runs/iam_init/log.csv` and `curves.png`. The
test numbers are in `test_metrics_*.json`. The comparison between the two is itself a result for the
paper: does English handwriting pre-training help on Modi, and by how much?